# Dataset Preparation and Filtering

**Importing polars**

for lazy loading as the dataset is huge

In [1]:
import polars as pl

* Loads the daily transaction dataset
* Loads the global target label dataset

In [2]:
daily = pl.read_parquet('../data/raw/chain/daily_filtered.parquet')
targets = pl.read_parquet('../data/raw/chain/targets_global.parquet')

**Explore the Daily and target dataset**
* Prints dataset shapes
* Prints data types
* Shows date range
* Displays scam class balance
* Shows how many addresses overlap between datasets

In [3]:
print("\ndaily shape:", daily.shape)
print("\ndaily column types:", daily.schema)
print("\nDaily Date Range:", daily['day'].min(), "-", daily['day'].max())

print("\ntargets shape:", targets.shape)
print("\ntargets schema:", targets.schema)


daily shape: (42946110, 31)

daily column types: Schema({'node_id': UInt64, 'address': String, 'day': Date, 'week': String, 'month': String, 'normal_sent_cnt': Int64, 'normal_recv_cnt': Int64, 'normal_total_cnt': Int64, 'normal_failed_cnt': Int64, 'normal_to_contract_cnt': Int64, 'normal_to_eoa_cnt': Int64, 'gas_used_sum': Int64, 'erc20_sent_cnt': Int64, 'erc20_recv_cnt': Int64, 'erc20_total_cnt': Int64, 'erc20_unique_tokens_sent': Int64, 'erc20_unique_tokens_recv': Int64, 'internal_out_cnt': Int64, 'internal_in_cnt': Int64, 'uniq_peers_cnt': Int64, 'uniq_contract_peers_cnt': Int64, 'uniq_eoa_peers_cnt': Int64, 'sessions_cnt': Int64, 'active_span_min': Int64, 'burst_max_tx_5m': Int64, 'eth_sent_sum': Decimal(precision=38, scale=9), 'eth_recv_sum': Decimal(precision=38, scale=9), 'eth_net_flow': Decimal(precision=38, scale=9), 'tx_fee_eth_sum': Decimal(precision=38, scale=9), 'internal_out_value_eth_sum': Decimal(precision=38, scale=9), 'internal_in_value_eth_sum': Decimal(precision=38

In [4]:
print("\nclass balance:", targets.group_by('is_scam').len())
print("\ncontract count", targets.group_by('is_contract').len())

scam_eoa = targets.filter((pl.col("is_scam") == 1) & (pl.col("is_contract") == 0))
print("\nScam EOA count (is_scam=1 & is_contract=0):", scam_eoa.height)

# Overlap of scam EOAs (is_scam=1 & is_contract=0) with daily data
scam_eoa_overlap = daily.select("address").unique().join(
    scam_eoa.select("address").unique(), on="address", how="inner")

print("\nOverlap scam EOA addresses with daily data:", scam_eoa_overlap["address"].n_unique())

print("\nunique addresses in targets:", targets['address'].n_unique())

# Cross-check overlap
overlap = daily.select('address').unique().join(
    targets.select('address').unique(), on='address', how='inner'
)
print("\noverlap addresses:", overlap['address'].n_unique())


class balance: shape: (2, 2)
┌─────────┬───────┐
│ is_scam ┆ len   │
│ ---     ┆ ---   │
│ i8      ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 60086 │
│ 1       ┆ 54653 │
└─────────┴───────┘

contract count shape: (2, 2)
┌─────────────┬───────┐
│ is_contract ┆ len   │
│ ---         ┆ ---   │
│ i8          ┆ u32   │
╞═════════════╪═══════╡
│ 1           ┆ 69698 │
│ 0           ┆ 45041 │
└─────────────┴───────┘

Scam EOA count (is_scam=1 & is_contract=0): 14982

Overlap scam EOA addresses with daily data: 12893

unique addresses in targets: 114739

overlap addresses: 85147


**Analyze date columns**

In [5]:
print(daily[['day', 'week', 'month']].head())

shape: (5, 3)
┌────────────┬──────────┬─────────┐
│ day        ┆ week     ┆ month   │
│ ---        ┆ ---      ┆ ---     │
│ date       ┆ str      ┆ str     │
╞════════════╪══════════╪═════════╡
│ 2017-07-28 ┆ 2017-W30 ┆ 2017-07 │
│ 2017-07-28 ┆ 2017-W30 ┆ 2017-07 │
│ 2017-07-28 ┆ 2017-W30 ┆ 2017-07 │
│ 2017-07-28 ┆ 2017-W30 ┆ 2017-07 │
│ 2017-07-28 ┆ 2017-W30 ┆ 2017-07 │
└────────────┴──────────┴─────────┘


**Create Filtered Files - scam**
* Converts Decimal columns to Float
* filter scam and eoa addresses
* keep only required columns
* Filters target labels to match available daily data
* Saves both filtered outputs into /processed/chain/


In [6]:
# convert Decimal columns to Float64
decimal_cols = [
    'eth_sent_sum',
    'eth_recv_sum',
    'eth_net_flow'
]

daily = daily.with_columns([pl.col(col).cast(pl.Float64) for col in decimal_cols])

In [7]:
# Filter scam EOAs only (is_scam=1 AND is_contract=0)
scam_targets = targets.filter((pl.col("is_scam") == 1) & (pl.col("is_contract") == 0))

# Join daily data with scam EOA targets
scam_daily = daily.join(scam_targets.select("address"), on="address", how="inner")

In [8]:
keep_cols = [
    "address",
    "day",
    "normal_sent_cnt",
    "normal_recv_cnt",
    "normal_total_cnt",
    "eth_sent_sum",
    "eth_recv_sum",
    "eth_net_flow",
    "uniq_peers_cnt",
    "sessions_cnt",
    "active_span_min",
    "burst_max_tx_5m"
]

scam_daily = scam_daily.select(keep_cols)

**Check for feature inconsistencies**

In [9]:
# columns that indicate activity
check_cols = [
    "normal_sent_cnt",
    "normal_recv_cnt",
    "eth_sent_sum",
    "eth_recv_sum",
    "uniq_peers_cnt",
    "sessions_cnt",
    "active_span_min",
    "burst_max_tx_5m",
]

# Inconsistency rule: if normal_total_cnt == 0, then NO activity column should be > 0
inconsistent = ((pl.col("normal_total_cnt") == 0) & pl.any_horizontal([pl.col(c) > 0 for c in check_cols]))

# count inconsistent rows
num_inconsistent = scam_daily.select(inconsistent.sum()).item()
print(f"Inconsistent rows: {num_inconsistent}")

# keep only active days
daily_cleaned = scam_daily.filter(pl.col("normal_total_cnt") > 0)

print(f"Original rows: {scam_daily.height}")
print(f"Remaining rows: {daily_cleaned.height}")
print(f"Removed rows: {scam_daily.height - daily_cleaned.height}")

Inconsistent rows: 11571
Original rows: 7236555
Remaining rows: 455258
Removed rows: 6781297


In [10]:
scam_targets_cleaned = scam_targets.join(daily_cleaned.select("address").unique(), on="address", how="inner")

daily_cleaned.write_parquet("../data/processed/chain/scam_daily_cleaned.parquet")
scam_targets_cleaned.write_parquet("../data/processed/chain/scam_targets_global.parquet")

print("\nSaved scam_daily_cleaned.parquet")
print("Saved scam_targets_global.parquet")


Saved scam_daily_cleaned.parquet
Saved scam_targets_global.parquet


# Investigate cleaned data

In [11]:
import pandas as pd

In [12]:
scam_daily = pd.read_parquet('../data/processed/chain/scam_daily_cleaned.parquet')
scam_targets = pd.read_parquet('../data/processed/chain/scam_targets_global.parquet')

In [13]:
print("\nScam Daily Shape:", scam_daily.shape)
print("Unique Scam EOA Addresses (daily):",scam_daily["address"].nunique())
print("Scam Daily Date Range:",scam_daily["day"].min(),"-",scam_daily["day"].max())
print("\nScam Targets Shape:", scam_targets.shape)
print("Unique Scam EOA Addresses (targets):",scam_targets["address"].nunique())
print("\nScam Daily dtypes:")
print(scam_daily.dtypes)



Scam Daily Shape: (455258, 12)
Unique Scam EOA Addresses (daily): 12434
Scam Daily Date Range: 2016-05-09 - 2025-08-17

Scam Targets Shape: (12434, 4)
Unique Scam EOA Addresses (targets): 12434

Scam Daily dtypes:
address              object
day                  object
normal_sent_cnt       int64
normal_recv_cnt       int64
normal_total_cnt      int64
eth_sent_sum        float64
eth_recv_sum        float64
eth_net_flow        float64
uniq_peers_cnt        int64
sessions_cnt          int64
active_span_min       int64
burst_max_tx_5m       int64
dtype: object


#### **Scam types**
check and group the scam types that is available in the dataset

In [14]:
import pandas as pd

In [15]:
scam_targets = pd.read_parquet('../data/processed/chain/scam_targets_global.parquet')
addr_labels = pd.read_csv('../data/raw/chain/addr_labels_balanced.csv')

In [16]:
addr_labels = addr_labels[["address", "description"]]

scam_with_types = scam_targets.merge(addr_labels, on="address", how="left")

total_addresses = scam_with_types["address"].nunique()
print("Total scam addresses:", total_addresses)

def map_scam_category(desc):
    if pd.isna(desc):
        return "Unknown"
    d = desc.lower()
    if "phish" in d:
        return "Phishing"
    elif "rug" in d:
        return "Rug Pull"
    elif "giveaway" in d:
        return "Giveaway Scam"
    elif "hack" in d:
        return "Hack / Exploit"
    elif "exploit" in d:
        return "Hack / Exploit"
    elif "ico" in d:
        return "Fake ICO"
    elif "impersonat" in d:
        return "Impersonation"
    else:
        return "Other"

scam_with_types["scam_category"] = scam_with_types["description"].apply(map_scam_category)

category_stats = (
    scam_with_types
    .groupby("scam_category")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(category_stats)

Total scam addresses: 12434
    scam_category  count
5        Phishing   6504
2  Hack / Exploit   4093
4           Other   1813
0        Fake ICO     11
6        Rug Pull     10
1   Giveaway Scam      1
3   Impersonation      1
7         Unknown      1
